In [4]:
import pdfplumber
import re
import os
import glob
import pandas as pd

class UberReportExtractor:
    def __init__(self, file_path):
        self.file_path = file_path
        self.full_text = ""
        self.page1_text = ""
        self.texto_plano = "" 
        self.data = {
            "driver_name": "Roman Yair Ortega",
            "weekly_period": "Not found",
            "total_trips": 0,
            "connected_hours_decimal": 0.0,
            "gross_amount": 0.0,
            "tips_amount": 0.0,
            "incentives_amount": 0.0,
            "taxes_amount": 0.0
        }

    def load_pdf(self):
        try:
            with pdfplumber.open(self.file_path) as pdf:
                for index, page in enumerate(pdf.pages):
                    text = page.extract_text()
                    if text:
                        clean_text = text.replace("\u0000", "")
                        self.full_text += clean_text + "\n"
                        if index == 0:
                            self.page1_text = clean_text
            
            # Texto aplastado para forzar la adyacencia
            self.texto_plano = " ".join(self.full_text.split())
            return True
        except Exception as e:
            print(f"❌ Error loading PDF: {e}")
            return False

    def parse_data(self):
        # 1. Fechas 
        match_date = re.search(r"(\d{2}/\d{2}/\d{4}).*?(\d{2}/\d{2}/\d{4})", self.page1_text)
        if match_date: 
            self.data["weekly_period"] = f"Del {match_date.group(1)} al {match_date.group(2)}"
        else:
            match_date_alt = re.search(r"(\d{1,2}\s+[a-z]{3}\s+\d{4}.*?-.*?\d{4})", self.page1_text, re.IGNORECASE)
            if match_date_alt:
                self.data["weekly_period"] = re.sub(r"\s+\d{1,2}\s+[ap]\.\s+m\.", "", match_date_alt.group(1)).strip()

        # 2. Viajes
        match_trips = re.search(r"viajes completados[^\d]*(\d+)", self.full_text, re.IGNORECASE)
        if match_trips: self.data["total_trips"] = int(match_trips.group(1))
            
# 3. Tiempo (Cambiamos a .*? para que logre saltar la trampa del "P2+P3")
        match_time = re.search(r"Tiempo de trabajo efectivo.*?(\d+)\s*h.*?(\d+)\s*m", self.texto_plano, re.IGNORECASE)
        if match_time:
            self.data["connected_hours_decimal"] = round(int(match_time.group(1)) + (int(match_time.group(2)) / 60.0), 2)
        # =========================================================
        # 💰 CONVERSOR DE DINERO INTELIGENTE
        # =========================================================
        def to_float(val_str):
            val_str = re.sub(r'[^\d\.,]', '', val_str)
            if not val_str: return 0.0
            if ',' in val_str and '.' in val_str:
                if val_str.rfind(',') > val_str.rfind('.'): 
                    val_str = val_str.replace('.', '').replace(',', '.')
                else:
                    val_str = val_str.replace(',', '')
            elif ',' in val_str:
                if len(val_str) > 3 and val_str[-3] == ',':
                    val_str = val_str.replace(',', '.')
                else:
                    val_str = val_str.replace(',', '')
            return float(val_str)

        # 🌟 EL CANDADO DE ADYACENCIA ESTRICTA (\s+)
        # Obliga a que el número esté PEGADO a la frase, ignorando subtítulos engañosos.
        # Soporta los nombres nuevos y viejos de Uber.
        
        match_gross = re.search(r"(?:Importe bruto que has generado|Monto bruto generado)\s+\$?\s*([\d\.,]+[\.,]\d{2})", self.texto_plano, re.IGNORECASE)
        if match_gross: self.data["gross_amount"] = to_float(match_gross.group(1))

        match_tips = re.search(r"(?:Propinas dadas por los usuarios|Propinas otorgadas)\s+\$?\s*([\d\.,]+[\.,]\d{2})", self.texto_plano, re.IGNORECASE)
        if match_tips: self.data["tips_amount"] = to_float(match_tips.group(1))

# 6. Incentivos (Agregamos "Otras ganancias" al diccionario de Uber)
        match_incentives = re.search(r"(?:Otras ganancias|Ganancias adicionales|promoción y fomento)\s*\$?\s*([\d\.,]+[\.,]\d{2})", self.texto_plano, re.IGNORECASE)
        if match_incentives: self.data["incentives_amount"] = to_float(match_incentives.group(1))

        match_taxes = re.search(r"Impuestos\s*(?:-?\s*\$?|-)\s*([\d\.,]+[\.,]\d{2})", self.texto_plano, re.IGNORECASE)
        if match_taxes: self.data["taxes_amount"] = -abs(to_float(match_taxes.group(1)))




In [7]:
# 1. Definir la ruta
ruta_prueba = "Uber_Reports/Abril_2025/resumen_7 de abr, 4_00.pdf"

# 2. Instanciar y Procesar
extractor = UberReportExtractor(file_path=ruta_prueba)

if extractor.load_pdf():
    extractor.parse_data()
    
    # 3. Mapear datos para el formato de impresión
    d = extractor.data
    
    # Formatear el tiempo de decimal a HHh MMm
    horas = int(d['connected_hours_decimal'])
    minutos = int((d['connected_hours_decimal'] - horas) * 60)
    formatted_time = f"{horas}h {minutos}m"
    
    # Formatear moneda (moneda local con comas y 2 decimales)
    f_gross = f"${d['gross_amount']:,.2f}"
    f_tips  = f"${d['tips_amount']:,.2f}"
    f_inc   = f"${d['incentives_amount']:,.2f}"
    # Los impuestos ya vienen negativos en tu lógica, usamos abs() para el signo en el print
    f_taxes = f"-${abs(d['taxes_amount']):,.2f}"

    # 4. IMPRESIÓN CON TU FORMATO SOLICITADO
    print("-" * 50)
    print("RESUMEN SEMANAL UBER 📊")
    print("-" * 50)
    print(f" 👤 CONDUCTOR:             {d['driver_name']}")
    print(f" 📅 SEMANA:                {d['weekly_period']}")
    print(f" 🚗 VIAJES COMPLETADOS:    {d['total_trips']}")
    print(f" ⏱️ TIEMPO CONECTADO:      {formatted_time}")
    print("-" * 50)
    print("DESGLOSE FINANCIERO")
    print(f" 💵 MONTO BRUTO GENERADO:  {f_gross}")
    print(f" 🪙 PROPINAS:              {f_tips}")
    print(f" 🎁 INCENTIVOS:            {f_inc}")
    print(f" 💸 IMPUESTOS:             {f_taxes}")
    print("-" * 50)
else:
    print("❌ Error: No se pudo leer el archivo.")

--------------------------------------------------
RESUMEN SEMANAL UBER 📊
--------------------------------------------------
 👤 CONDUCTOR:             Roman Yair Ortega
 📅 SEMANA:                Del 31/03/2025 al 07/04/2025
 🚗 VIAJES COMPLETADOS:    64
 ⏱️ TIEMPO CONECTADO:      20h 19m
--------------------------------------------------
DESGLOSE FINANCIERO
 💵 MONTO BRUTO GENERADO:  $3,329.24
 🪙 PROPINAS:              $0.00
 🎁 INCENTIVOS:            $0.00
 💸 IMPUESTOS:             -$270.25
--------------------------------------------------


In [6]:
# Para ver el texto "aplastado" que procesan los Regex de dinero y tiempo:
print("\n--- VISTA PREVIA DEL TEXTO PLANO ---")
print(extractor.texto_plano[:500]) # Ver los primeros 500 caracteres

# Para ver la página 1 (donde se busca el nombre y las fechas):
print("\n--- VISTA PREVIA DE LA PÁGINA 1 ---")
print(extractor.page1_text[:500])


--- VISTA PREVIA DEL TEXTO PLANO ---
Extracto semanal 31/03/2025 400 - 07/04/2025 400 Tiempo efectivamente laborado Tiempo efectivamente laborado - Uber, servicios gestionados y administrados por Lieber BV, S. de R.L. de C.V. El tiempo de trabajo efectivo va desde el momento de la aceptación de un viaje hasta el momento de finalización del mismo. Número total de viajes completados 64 Tiempo de trabajo efectivo = P2+P3 20 h 20 m Tiempo de espera = P2+P3 20 h 20 m Tu Monto Bruto Generado incluye el pago proporcional del día de descan

--- VISTA PREVIA DE LA PÁGINA 1 ---
Extracto semanal
31/03/2025 400 - 07/04/2025 400
Tiempo efectivamente laborado
Tiempo efectivamente laborado - Uber, servicios gestionados y administrados por Lieber BV, S. de R.L. de
C.V.
El tiempo de trabajo efectivo va desde el momento de la aceptación de un viaje hasta el
momento de finalización del mismo.
Número total de viajes completados 64
Tiempo de trabajo efectivo = P2+P3 20 h 20 m
Tiempo de espera = P2+P3 20 h

In [8]:
# 1. Definimos una ruta diferente (Diciembre 2024)
ruta_diciembre = "Uber_Reports/Diciembre_2024/resumen_16 de dic, 4_00.pdf"

# 2. Instanciamos el extractor
test_dec = UberReportExtractor(file_path=ruta_diciembre)

print(f"🔍 Analizando: {ruta_diciembre}...")

if test_dec.load_pdf():
    test_dec.parse_data()
    
    # 3. Preparación de variables para el formato solicitado
    d = test_dec.data
    
    # Conversión de horas decimales a formato legible
    h = int(d['connected_hours_decimal'])
    m = int((d['connected_hours_decimal'] - h) * 60)
    formatted_time = f"{h}h {m}m"
    
    # Formateo de moneda
    f_gross = f"${d['gross_amount']:,.2f}"
    f_tips  = f"${d['tips_amount']:,.2f}"
    f_inc   = f"${d['incentives_amount']:,.2f}"
    f_taxes = f"-${abs(d['taxes_amount']):,.2f}"

    # 4. IMPRESIÓN DEL RESUMEN 📊
    print("-" * 50)
    print("RESUMEN SEMANAL UBER 📊")
    print("-" * 50)
    print(f" 👤 CONDUCTOR:             {d['driver_name']}")
    print(f" 📅 SEMANA:                {d['weekly_period']}")
    print(f" 🚗 VIAJES COMPLETADOS:    {d['total_trips']}")
    print(f" ⏱️ TIEMPO CONECTADO:      {formatted_time}")
    print("-" * 50)
    print("DESGLOSE FINANCIERO")
    print(f" 💵 MONTO BRUTO GENERADO:  {f_gross}")
    print(f" 🪙 PROPINAS:              {f_tips}")
    print(f" 🎁 INCENTIVOS:            {f_inc}")
    print(f" 💸 IMPUESTOS:             {f_taxes}")
    print("-" * 50)
else:
    print(f"❌ No se encontró el archivo en: {ruta_diciembre}")

🔍 Analizando: Uber_Reports/Diciembre_2024/resumen_16 de dic, 4_00.pdf...
--------------------------------------------------
RESUMEN SEMANAL UBER 📊
--------------------------------------------------
 👤 CONDUCTOR:             Roman Yair Ortega
 📅 SEMANA:                Del 09/12/2024 al 16/12/2024
 🚗 VIAJES COMPLETADOS:    70
 ⏱️ TIEMPO CONECTADO:      38h 45m
--------------------------------------------------
DESGLOSE FINANCIERO
 💵 MONTO BRUTO GENERADO:  $9,819.44
 🪙 PROPINAS:              $80.00
 🎁 INCENTIVOS:            $0.00
 💸 IMPUESTOS:             -$1,033.72
--------------------------------------------------


In [9]:
import os

# 1. Definir la carpeta objetivo
carpeta_febrero = "Uber_Reports/Febrero_2025"

# 2. Inicializar acumuladores para el Gran Total del mes
total_mes_viajes = 0
total_mes_dinero = 0.0
total_mes_horas = 0.0
archivos_procesados = 0

print(f"📂 Iniciando procesamiento de la carpeta: {carpeta_febrero}\n")

# 3. Bucle para procesar cada archivo
if os.path.exists(carpeta_febrero):
    # Ordenamos los archivos para que aparezcan en orden cronológico
    for archivo in sorted(os.listdir(carpeta_febrero)):
        if archivo.lower().endswith(".pdf"):
            ruta_completa = os.path.join(carpeta_febrero, archivo)
            
            extractor = UberReportExtractor(file_path=ruta_completa)
            
            if extractor.load_pdf():
                extractor.parse_data()
                d = extractor.data
                
                # Acumular para el Gran Total
                total_mes_viajes += d['total_trips']
                total_mes_dinero += d['gross_amount']
                total_mes_horas += d['connected_hours_decimal']
                archivos_procesados += 1
                
                # Formateo de datos individuales
                h = int(d['connected_hours_decimal'])
                m = int((d['connected_hours_decimal'] - h) * 60)
                f_time = f"{h}h {m}m"
                
                # IMPRESIÓN DEL TICKET SEMANAL
                print("-" * 50)
                print(f"📄 ARCHIVO: {archivo}")
                print("-" * 50)
                print(f" 📅 SEMANA:                {d['weekly_period']}")
                print(f" 🚗 VIAJES COMPLETADOS:    {d['total_trips']}")
                print(f" ⏱️ TIEMPO CONECTADO:      {f_time}")
                print(f" 💵 MONTO BRUTO:           ${d['gross_amount']:,.2f}")
                print(f" 💸 IMPUESTOS:             -${abs(d['taxes_amount']):,.2f}")
                print("-" * 50)
                print("\n")

    # 4. RESUMEN FINAL DEL MES
    print("=" * 50)
    print("🏆 RESUMEN TOTAL: FEBRERO 2025")
    print("=" * 50)
    print(f" 📂 Reportes procesados:    {archivos_procesados}")
    print(f" 🚗 Total Viajes:           {total_mes_viajes}")
    print(f" ⏱️ Total Horas:            {int(total_mes_horas)}h {int((total_mes_horas % 1)*60)}m")
    print(f" 💰 Ingreso Bruto Total:    ${total_mes_dinero:,.2f}")
    print(f" 📈 Promedio por Viaje:     ${(total_mes_dinero/total_mes_viajes if total_mes_viajes > 0 else 0):,.2f}")
    print("=" * 50)

else:
    print(f"❌ La carpeta '{carpeta_febrero}' no existe. Verifica la ruta.")

📂 Iniciando procesamiento de la carpeta: Uber_Reports/Febrero_2025

--------------------------------------------------
📄 ARCHIVO: resumen_10 de feb, 4_00.pdf
--------------------------------------------------
 📅 SEMANA:                Del 03/02/2025 al 10/02/2025
 🚗 VIAJES COMPLETADOS:    61
 ⏱️ TIEMPO CONECTADO:      18h 21m
 💵 MONTO BRUTO:           $3,107.00
 💸 IMPUESTOS:             -$228.81
--------------------------------------------------


--------------------------------------------------
📄 ARCHIVO: resumen_17 de feb, 4_00.pdf
--------------------------------------------------
 📅 SEMANA:                Del 10/02/2025 al 17/02/2025
 🚗 VIAJES COMPLETADOS:    106
 ⏱️ TIEMPO CONECTADO:      36h 25m
 💵 MONTO BRUTO:           $6,032.38
 💸 IMPUESTOS:             -$527.49
--------------------------------------------------


--------------------------------------------------
📄 ARCHIVO: resumen_24 de feb, 4_00.pdf
--------------------------------------------------
 📅 SEMANA:            

In [10]:
import os

# 1. Definimos la carpeta objetivo
carpeta_objetivo = "Uber_Reports/Febrero_2025"

# 2. Inicializamos contadores para el resumen mensual
stats_mes = {
    'viajes': 0,
    'ingresos': 0.0,
    'tiempo_horas': 0.0,
    'impuestos': 0.0,
    'conteo_archivos': 0
}

print(f"📂 Procesando archivos en: {carpeta_objetivo}...")
print("-" * 50)

if os.path.exists(carpeta_objetivo):
    # Listar archivos y ordenarlos
    archivos = [f for f in os.listdir(carpeta_objetivo) if f.lower().endswith('.pdf')]
    
    for archivo in sorted(archivos):
        ruta_pdf = os.path.join(carpeta_objetivo, archivo)
        extractor = UberReportExtractor(file_path=ruta_pdf)
        
        if extractor.load_pdf():
            extractor.parse_data()
            d = extractor.data
            
            # Acumular estadísticas
            stats_mes['viajes'] += d['total_trips']
            stats_mes['ingresos'] += d['gross_amount']
            stats_mes['tiempo_horas'] += d['connected_hours_decimal']
            stats_mes['impuestos'] += d['taxes_amount']
            stats_mes['conteo_archivos'] += 1
            
            # Formateo de tiempo individual
            h = int(d['connected_hours_decimal'])
            m = int((d['connected_hours_decimal'] - h) * 60)
            
            # 3. IMPRESIÓN DEL RESUMEN SEMANAL
            print("RESUMEN SEMANAL UBER 📊")
            print(f"📄 Archivo: {archivo}")
            print("-" * 50)
            print(f" 👤 CONDUCTOR:             {d['driver_name']}")
            print(f" 📅 SEMANA:                {d['weekly_period']}")
            print(f" 🚗 VIAJES COMPLETADOS:    {d['total_trips']}")
            print(f" ⏱️ TIEMPO CONECTADO:      {h}h {m}m")
            print("-" * 50)
            print("DESGLOSE FINANCIERO")
            print(f" 💵 MONTO BRUTO GENERADO:  ${d['gross_amount']:,.2f}")
            print(f" 🪙 PROPINAS:              ${d['tips_amount']:,.2f}")
            print(f" 🎁 INCENTIVOS:            ${d['incentives_amount']:,.2f}")
            print(f" 💸 IMPUESTOS:             -${abs(d['taxes_amount']):,.2f}")
            print("-" * 50)
            print("\n")

    # 4. GRAN RESUMEN MENSUAL
    if stats_mes['conteo_archivos'] > 0:
        h_total = int(stats_mes['tiempo_horas'])
        m_total = int((stats_mes['tiempo_horas'] - h_total) * 60)
        
        print("=" * 50)
        print("🏆 RESUMEN TOTAL: FEBRERO 2025")
        print("=" * 50)
        print(f" 📂 Reportes analizados:   {stats_mes['conteo_archivos']}")
        print(f" 🚗 Viajes totales:        {stats_mes['viajes']}")
        print(f" ⏱️ Tiempo total:          {h_total}h {m_total}m")
        print(f" 💰 Ingreso Bruto:         ${stats_mes['ingresos']:,.2f}")
        print(f" 💸 Impuestos totales:     -${abs(stats_mes['impuestos']):,.2f}")
        print(f" 📈 Ganancia promedio/viaje: ${(stats_mes['ingresos']/stats_mes['viajes'] if stats_mes['viajes'] > 0 else 0):,.2f}")
        print("=" * 50)
else:
    print(f"⚠️ La carpeta '{carpeta_objetivo}' no existe en el sistema.")

📂 Procesando archivos en: Uber_Reports/Febrero_2025...
--------------------------------------------------
RESUMEN SEMANAL UBER 📊
📄 Archivo: resumen_10 de feb, 4_00.pdf
--------------------------------------------------
 👤 CONDUCTOR:             Roman Yair Ortega
 📅 SEMANA:                Del 03/02/2025 al 10/02/2025
 🚗 VIAJES COMPLETADOS:    61
 ⏱️ TIEMPO CONECTADO:      18h 21m
--------------------------------------------------
DESGLOSE FINANCIERO
 💵 MONTO BRUTO GENERADO:  $3,107.00
 🪙 PROPINAS:              $0.00
 🎁 INCENTIVOS:            $0.00
 💸 IMPUESTOS:             -$228.81
--------------------------------------------------


RESUMEN SEMANAL UBER 📊
📄 Archivo: resumen_17 de feb, 4_00.pdf
--------------------------------------------------
 👤 CONDUCTOR:             Roman Yair Ortega
 📅 SEMANA:                Del 10/02/2025 al 17/02/2025
 🚗 VIAJES COMPLETADOS:    106
 ⏱️ TIEMPO CONECTADO:      36h 25m
--------------------------------------------------
DESGLOSE FINANCIERO
 💵 MONTO BR

In [12]:
import os
import re
import pandas as pd

def consolidar_analisis_anual(root_folder):
    resumen_anual = {}
    meses_orden = {
        'enero': 1, 'febrero': 2, 'marzo': 3, 'abril': 4, 'mayo': 5, 'junio': 6,
        'julio': 7, 'agosto': 8, 'septiembre': 9, 'octubre': 10, 'noviembre': 11, 'diciembre': 12
    }
    
    print("🚀 Iniciando Escaneo Global de Reportes...")
    
    # 1. RASTREO DE CARPETAS
    for root, dirs, files in os.walk(root_folder):
        folder_name = os.path.basename(root).lower()
        mes_detectado = next((m for m in meses_orden if m in folder_name), None)
        anio_match = re.search(r"202\d", folder_name)
        
        if mes_detectado and anio_match:
            anio = anio_match.group()
            # Creamos una clave para ordenar: "2025-02 (Febrero)"
            clave_mes = f"{anio}-{meses_orden[mes_detectado]:02d} ({mes_detectado.capitalize()})"
            
            if clave_mes not in resumen_anual:
                resumen_anual[clave_mes] = {
                    'viajes': 0, 'bruto': 0.0, 'horas': 0.0, 'impuestos': 0.0, 'reportes': 0
                }
            
            for file in files:
                if file.lower().endswith(".pdf"):
                    path = os.path.join(root, file)
                    ex = UberReportExtractor(file_path=path)
                    
                    if ex.load_pdf():
                        ex.parse_data()
                        d = ex.data
                        
                        # Acumulación segura
                        resumen_anual[clave_mes]['viajes'] += d['total_trips']
                        resumen_anual[clave_mes]['bruto'] += d['gross_amount']
                        resumen_anual[clave_mes]['horas'] += d['connected_hours_decimal']
                        resumen_anual[clave_mes]['impuestos'] += d['taxes_amount']
                        resumen_anual[clave_mes]['reportes'] += 1

    # 2. IMPRESIÓN Y BALANCE
    print("\n" + "="*65)
    print("📊 ESTADO DE RESULTADOS CONSOLIDADO")
    print("="*65)
    
    gran_total_viajes = 0
    gran_total_dinero = 0.0
    gran_total_horas = 0.0
    
    # Ordenar por la clave (Año-Mes)
    for mes in sorted(resumen_anual.keys()):
        m_data = resumen_anual[mes]
        gran_total_viajes += m_data['viajes']
        gran_total_dinero += m_data['bruto']
        gran_total_horas += m_data['horas']
        
        # Formato corregido con :.2f para evitar el ValueError
        print(f"📅 {mes}:")
        print(f"   🚗 Viajes: {m_data['viajes']} | 💰 Bruto: ${m_data['bruto']:,.2f}")
        print(f"   ⏱️  Horas: {m_data['horas']:.2f}h | 💸 Impuestos: ${abs(m_data['impuestos']):,.2f}")
        print("-" * 55)

    # 3. GRAN BALANCE ANUAL FINAL
    print("\n" + "🏆" + " BALANCE TOTAL ACUMULADO ".center(63, "="))
    if gran_total_viajes > 0 and gran_total_horas > 0:
        print(f" 📈 Total Viajes Realizados:   {gran_total_viajes}")
        print(f" 💰 Ingreso Bruto Acumulado:  ${gran_total_dinero:,.2f}")
        print(f" ⏱️  Tiempo Total Conectado:   {gran_total_horas:,.2f} horas")
        print(f" 💵 Promedio por Viaje:       ${(gran_total_dinero/gran_total_viajes):,.2f}")
        print(f" 🕒 Ganancia Bruta por Hora:  ${(gran_total_dinero/gran_total_horas):,.2f}")
    else:
        print("⚠️ No se procesaron suficientes datos para el balance final.")
    print("="*65)

# Ejecutar el análisis global
consolidar_analisis_anual('Uber_Reports')

🚀 Iniciando Escaneo Global de Reportes...

📊 ESTADO DE RESULTADOS CONSOLIDADO
📅 2024-12 (Diciembre):
   🚗 Viajes: 368 | 💰 Bruto: $36,761.48
   ⏱️  Horas: 161.57h | 💸 Impuestos: $4,571.94
-------------------------------------------------------
📅 2025-01 (Enero):
   🚗 Viajes: 563 | 💰 Bruto: $40,048.96
   ⏱️  Horas: 213.35h | 💸 Impuestos: $5,484.81
-------------------------------------------------------
📅 2025-02 (Febrero):
   🚗 Viajes: 468 | 💰 Bruto: $28,241.32
   ⏱️  Horas: 160.39h | 💸 Impuestos: $2,945.79
-------------------------------------------------------
📅 2025-03 (Marzo):
   🚗 Viajes: 492 | 💰 Bruto: $25,840.97
   ⏱️  Horas: 150.13h | 💸 Impuestos: $1,961.67
-------------------------------------------------------
📅 2025-04 (Abril):
   🚗 Viajes: 379 | 💰 Bruto: $18,718.25
   ⏱️  Horas: 106.41h | 💸 Impuestos: $1,633.59
-------------------------------------------------------
📅 2025-05 (Mayo):
   🚗 Viajes: 370 | 💰 Bruto: $21,275.55
   ⏱️  Horas: 115.63h | 💸 Impuestos: $2,020.18
-------

In [16]:
import os

# 1. Definir la carpeta objetivo
carpeta_marzo = "Uber_Reports/Marzo_2025"

# 2. Inicializar acumuladores para el Gran Total del mes
total_mes_viajes = 0
total_mes_dinero = 0.0
total_mes_horas = 0.0
archivos_procesados = 0

print(f"📂 Iniciando procesamiento de la carpeta: {carpeta_marzo}\n")

# 3. Bucle para procesar cada archivo
if os.path.exists(carpeta_febrero):
    # Ordenamos los archivos para que aparezcan en orden cronológico
    for archivo in sorted(os.listdir(carpeta_febrero)):
        if archivo.lower().endswith(".pdf"):
            ruta_completa = os.path.join(carpeta_marzo, archivo)
            
            extractor = UberReportExtractor(file_path=ruta_completa)
            
            if extractor.load_pdf():
                extractor.parse_data()
                d = extractor.data
                
                # Acumular para el Gran Total
                total_mes_viajes += d['total_trips']
                total_mes_dinero += d['gross_amount']
                total_mes_horas += d['connected_hours_decimal']
                f_tips  = f"${d['tips_amount']:,.2f}"
                archivos_procesados += 1
                
                # Formateo de datos individuales
                h = int(d['connected_hours_decimal'])
                m = int((d['connected_hours_decimal'] - h) * 60)
                f_time = f"{h}h {m}m"
                
                # IMPRESIÓN DEL TICKET SEMANAL
                print("-" * 50)
                print(f"📄 ARCHIVO: {archivo}")
                print("-" * 50)
                print(f" 📅 SEMANA:                {d['weekly_period']}")
                print(f" 🚗 VIAJES COMPLETADOS:    {d['total_trips']}")
                print(f" ⏱️ TIEMPO CONECTADO:      {f_time}")
                print(f" 🪙 PROPINAS:              {f_tips}")
                print(f" 💵 MONTO BRUTO:           ${d['gross_amount']:,.2f}")
                print(f" 💸 IMPUESTOS:             -${abs(d['taxes_amount']):,.2f}")
                print("-" * 50)
                print("\n")

    # 4. RESUMEN FINAL DEL MES
    print("=" * 50)
    print("🏆 RESUMEN TOTAL: MARZO 2025")
    print("=" * 50)
    print(f" 📂 Reportes procesados:    {archivos_procesados}")
    print(f" 🚗 Total Viajes:           {total_mes_viajes}")
    print(f" ⏱️ Total Horas:            {int(total_mes_horas)}h {int((total_mes_horas % 1)*60)}m")
    print(f" 💰 Ingreso Bruto Total:    ${total_mes_dinero:,.2f}")
    print(f" 📈 Promedio por Viaje:     ${(total_mes_dinero/total_mes_viajes if total_mes_viajes > 0 else 0):,.2f}")
    print("=" * 50)

else:
    print(f"❌ La carpeta '{carpeta_marzo}' no existe. Verifica la ruta.")

📂 Iniciando procesamiento de la carpeta: Uber_Reports/Marzo_2025

--------------------------------------------------
📄 ARCHIVO: resumen_10 de mar, 4_00.pdf
--------------------------------------------------
 📅 SEMANA:                Del 03/03/2025 al 10/03/2025
 🚗 VIAJES COMPLETADOS:    80
 ⏱️ TIEMPO CONECTADO:      26h 11m
 🪙 PROPINAS:              $15.00
 💵 MONTO BRUTO:           $4,567.21
 💸 IMPUESTOS:             -$260.62
--------------------------------------------------


--------------------------------------------------
📄 ARCHIVO: resumen_17 de mar, 4_00.pdf
--------------------------------------------------
 📅 SEMANA:                Del 10/03/2025 al 17/03/2025
 🚗 VIAJES COMPLETADOS:    105
 ⏱️ TIEMPO CONECTADO:      31h 1m
 🪙 PROPINAS:              $0.00
 💵 MONTO BRUTO:           $5,249.90
 💸 IMPUESTOS:             -$391.83
--------------------------------------------------


--------------------------------------------------
📄 ARCHIVO: resumen_24 de mar, 4_00.pdf
-----------

In [25]:
import json
import pandas as pd

def segmentar_json_por_timestamp(path_json):
    with open(path_json, 'r', encoding='utf-8') as f:
        datos = json.load(f)
    
    segmentado = {}
    procesados = 0
    
    for viaje in datos:
        # 1. Extraer Fecha desde el Timestamp de Unix
        ts = viaje.get('recognizedAt')
        if not ts:
            continue
            
        try:
            # Convertimos segundos a fecha (pd.to_datetime con unit='s')
            fecha_dt = pd.to_datetime(ts, unit='s')
            # Ajuste a hora local (opcional, restamos 6 horas para México)
            fecha_dt = fecha_dt - pd.Timedelta(hours=6) 
            
            mes_key = fecha_dt.strftime('%Y-%m')
            
            if mes_key not in segmentado:
                segmentado[mes_key] = []
            
            # 2. Limpieza del Monto (40,49 MXN -> 40.49)
            monto_raw = viaje.get('formattedTotal', '0')
            # Reemplazamos coma por punto y quitamos texto
            monto_str = monto_raw.replace('MXN', '').replace('\xa0', '').replace(',', '.').strip()
            monto_numeric = float(monto_str)
            
            # 3. Guardar datos enriquecidos
            viaje['fecha_dt'] = fecha_dt
            viaje['monto_numeric'] = monto_numeric
            viaje['tipo_viaje'] = "REAL" if monto_numeric > 30 else "CANCELACIÓN/ESPERA"
            
            segmentado[mes_key].append(viaje)
            procesados += 1
        except Exception as e:
            continue
            
    print(f"✅ ¡Éxito! Se recuperaron {procesados} viajes.")
    return segmentado

# Ejecutar
json_por_mes = segmentar_json_por_timestamp('viajes_uber_detallados.json')

✅ ¡Éxito! Se recuperaron 494 viajes.


In [23]:
with open('viajes_uber_detallados.json', 'r', encoding='utf-8') as f:
    datos = json.load(f)

print("Estructura del primer registro:")
import pprint
pprint.pprint(datos[0]) # Esto nos dirá los nombres reales de las llaves

Estructura del primer registro:
{'activityTitle': 'UberX',
 'breakdownDetails': None,
 'formattedTotal': '40,49\xa0MXN',
 'recognizedAt': 1740354379,
 'routing': {'deeplinkUrl': None,
             'webviewUrl': 'https://drivers.uber.com/earnings/trips/c7e312f0-a155-4c92-965d-004e8fb0281d'},
 'status': 'COMPLETED',
 'tripMetaData': {'dropOffAddress': 'Benito Juárez, Morelos, 58341 Michoacán, '
                                    'MX',
                  'formattedDistance': '3.36 km',
                  'formattedDuration': '14\xa0min 19\xa0seg',
                  'mapUrl': 'https://static-maps.uber.com/map?width=360&height=100&marker=lat%3A19.66204%24lng%3A-101.22142%24icon%3Ahttps%3A%2F%2Fd1a3f4spazzrp4.cloudfront.net%2Fmaps%2Fhelix%2Fcar-pickup-pin.png%24anchorX%3A1.0%24anchorY%3A0.5&marker=lat%3A19.64331%24lng%3A-101.23678%24icon%3Ahttps%3A%2F%2Fd1a3f4spazzrp4.cloudfront.net%2Fmaps%2Fhelix%2Fcar-dropoff-pin.png%24anchorX%3A1.0%24anchorY%3A0.5&polyline=color%3A0xFF2DBAE4%24width%3A4%24

In [27]:
import pandas as pd

lista_viajes = []

# Recorremos el diccionario que ya tienes segmentado
for mes, viajes in json_por_mes.items():
    for v in viajes:
        # 1. SEGURIDAD: Obtenemos el diccionario meta o uno vacío si no existe
        meta = v.get('tripMetaData') if v.get('tripMetaData') is not None else {}
        
        # 2. Extracción segura de datos
        lista_viajes.append({
            'Mes': mes,
            'Monto': v.get('monto_numeric', 0.0),
            'Tipo': v.get('tipo_viaje', 'DESCONOCIDO'),
            'Distancia': meta.get('formattedDistance', '0 km'), # Ya no falla si meta es None
            'Direccion': meta.get('pickupAddress', 'Sin dirección')
        })

df_json = pd.DataFrame(lista_viajes)

# 3. Resumen por Mes
resumen = df_json.groupby('Mes').agg({
    'Monto': ['count', 'sum'],
    'Tipo': lambda x: (x == 'REAL').sum()
})

resumen.columns = ['Total Capturados', 'Ingreso Capturado', 'Viajes Reales']
resumen['Cancelaciones/Esperas'] = resumen['Total Capturados'] - resumen['Viajes Reales']

print("📈 ANÁLISIS DE ACTIVIDAD JSON (CORREGIDO)")
print("=" * 65)
print(resumen)
print("-" * 65)
print(f"💰 TOTAL INGRESO MAPEADO: ${df_json['Monto'].sum():,.2f}")

📈 ANÁLISIS DE ACTIVIDAD JSON (CORREGIDO)
         Total Capturados  Ingreso Capturado  Viajes Reales  \
Mes                                                           
2025-02                30            1322.56             26   
2025-03                75            4451.02             64   
2025-04                60            2519.43             39   
2025-05                60            3533.38             48   
2025-06                59            4352.13             48   
2025-07                59            5188.93             51   
2025-08                74            4551.44             53   
2025-09                 3             275.84              1   
2025-11                 1             -21.00              0   
2025-12                59            5195.99             48   
2026-01                14            1569.11             10   

         Cancelaciones/Esperas  
Mes                             
2025-02                      4  
2025-03                     11  
2025-04

In [28]:
import pandas as pd
import re
import json

# 1. 🎯 ELIGE TU ARCHIVO DE PRUEBA AQUÍ
ruta_pdf_prueba = "Uber_Reports/Marzo_2025/resumen_31 de mar, 4_00.pdf" # Cámbialo por el archivo que gustes

print(f"🔍 1. ANALIZANDO PDF (VERDAD FINANCIERA): \n📂 {ruta_pdf_prueba}")
print("-" * 60)

extractor = UberReportExtractor(file_path=ruta_pdf_prueba)
if not extractor.load_pdf():
    print("❌ Error cargando el PDF. Verifica la ruta.")
else:
    extractor.parse_data()
    d_pdf = extractor.data
    periodo = d_pdf['weekly_period']
    viajes_pdf = d_pdf['total_trips']
    monto_pdf = d_pdf['gross_amount']
    
    print(f" 📅 Periodo extraído:     {periodo}")
    print(f" 🚗 Viajes (Contables):   {viajes_pdf}")
    print(f" 💵 Monto Bruto PDF:      ${monto_pdf:,.2f}")

    # 2. 🎯 EXTRAER FECHAS EXACTAS PARA EL FILTRO
    match_fechas = re.search(r"Del (\d{2}/\d{2}/\d{4}) al (\d{2}/\d{2}/\d{4})", periodo)
    if match_fechas:
        # Convertimos a formato fecha
        fecha_inicio = pd.to_datetime(match_fechas.group(1), format='%d/%m/%Y')
        # Sumamos 1 día a la fecha final para que incluya las 23:59 de ese último día
        fecha_fin = pd.to_datetime(match_fechas.group(2), format='%d/%m/%Y') + pd.Timedelta(days=1)
        
        print(f"\n🛰️ 2. BUSCANDO EN EL JSON (VERDAD GEOGRÁFICA):")
        print(f"   Buscando desde {fecha_inicio.date()} hasta {match_fechas.group(2)}...")
        print("-" * 60)
        
        # 3. 🎯 ESCANEAR JSON
        with open('viajes_uber_detallados.json', 'r', encoding='utf-8') as f:
            datos_json = json.load(f)
        
        viajes_semana = []
        for v in datos_json:
            ts = v.get('recognizedAt')
            if ts:
                # Ajuste horario a México (CST)
                fecha_dt = pd.to_datetime(ts, unit='s') - pd.Timedelta(hours=6)
                
                # Si el viaje ocurrió en esta semana, lo guardamos
                if fecha_inicio <= fecha_dt < fecha_fin:
                    monto_raw = v.get('formattedTotal', '0')
                    monto_str = str(monto_raw).replace('MXN', '').replace('\xa0', '').replace(',', '.').strip()
                    try:
                        monto_num = float(monto_str)
                    except:
                        monto_num = 0.0
                        
                    tipo = "REAL" if monto_num > 30 else "CANCELACIÓN/ESPERA"
                    viajes_semana.append({
                        'fecha': fecha_dt,
                        'monto': monto_num,
                        'tipo': tipo
                    })
        
        df_semana = pd.DataFrame(viajes_semana)
        
        # 4. 🎯 VEREDICTO DE PRECISIÓN
        if not df_semana.empty:
            viajes_json_total = len(df_semana)
            viajes_reales = len(df_semana[df_semana['tipo'] == 'REAL'])
            cancelaciones = len(df_semana[df_semana['tipo'] == 'CANCELACIÓN/ESPERA'])
            monto_json_total = df_semana['monto'].sum()
            
            print(f" 📊 Total capturados:     {viajes_json_total}")
            print(f" 🚗 Viajes Reales:        {viajes_reales}")
            print(f" ⚠️ Cancelaciones/Espera: {cancelaciones}")
            print(f" 💵 Monto Mapeado:        ${monto_json_total:,.2f}")
            
            print("\n⚖️ 3. VEREDICTO FINAL DE LA SEMANA:")
            print("=" * 60)
            
            tasa_captura = (viajes_json_total / viajes_pdf) * 100 if viajes_pdf > 0 else 0
            brecha_dinero = monto_pdf - monto_json_total
            
            print(f" 🎯 Tasa de Captura (JSON vs PDF): {tasa_captura:.1f}%")
            print(f" 📉 Brecha de Viajes:              Faltan {viajes_pdf - viajes_json_total} viajes en el mapa.")
            print(f" 💸 Brecha de Ingresos:            ${brecha_dinero:,.2f} no mapeados.")
            print("=" * 60)
            
            if cancelaciones > 0:
                print(f"💡 Nota: {cancelaciones} viajes de los {viajes_pdf} del PDF fueron detectados como simples interacciones financieras (Cancelaciones).")
        else:
            print(" ❌ No se encontraron viajes en el mapa para esta semana.")
    else:
        print(" ❌ El Regex no pudo leer la fecha de este PDF.")

🔍 1. ANALIZANDO PDF (VERDAD FINANCIERA): 
📂 Uber_Reports/Marzo_2025/resumen_31 de mar, 4_00.pdf
------------------------------------------------------------
 📅 Periodo extraído:     Del 24/03/2025 al 31/03/2025
 🚗 Viajes (Contables):   103
 💵 Monto Bruto PDF:      $4,826.11

🛰️ 2. BUSCANDO EN EL JSON (VERDAD GEOGRÁFICA):
   Buscando desde 2025-03-24 hasta 31/03/2025...
------------------------------------------------------------
 📊 Total capturados:     15
 🚗 Viajes Reales:        14
 ⚠️ Cancelaciones/Espera: 1
 💵 Monto Mapeado:        $840.05

⚖️ 3. VEREDICTO FINAL DE LA SEMANA:
 🎯 Tasa de Captura (JSON vs PDF): 14.6%
 📉 Brecha de Viajes:              Faltan 88 viajes en el mapa.
 💸 Brecha de Ingresos:            $3,986.06 no mapeados.
💡 Nota: 1 viajes de los 103 del PDF fueron detectados como simples interacciones financieras (Cancelaciones).
